# E-commerce Medallion Pipeline — Full Run

Runs **Bronze → Silver → Gold** from a [Databricks Repo](https://docs.databricks.com/repos/index.html) clone of:

`https://github.com/RAHUL9868/databricks-medallion-pipeline`

**Prerequisites**
- Repo attached to this cluster (Repos → Pull latest `main`)
- Cluster with Delta Lake (all-purpose or serverless)
- **Sample CSVs live under `{REPO_ROOT}/data`** (workspace path) — public DBFS `/FileStore` is often disabled on serverless
- Set `generate_sample_data` to `true` to create seed=42 CSVs in the repo `data/` folder automatically

**After this notebook:** build the SQL Dashboard using `src/dashboard/DASHBOARD_GUIDE.md`.

## 1. Configuration

Adjust widgets, then run the next cells in order.

In [ ]:
# Reset legacy widget value cached from older notebook versions (FileStore is disabled on serverless).
_legacy_path = dbutils.widgets.get("source_base_path").strip()
if "/FileStore/" in _legacy_path:
    dbutils.widgets.remove("source_base_path")

dbutils.widgets.text("schema_name", "ecommerce", "Hive schema / database")
dbutils.widgets.text(
    "source_base_path",
    "",
    "CSV directory (leave empty = {REPO_ROOT}/data; or UC volume path)",
)
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate seed=42 CSVs into repo data/")
dbutils.widgets.text("catalog", "", "Unity Catalog (optional, leave empty on CE)")
dbutils.widgets.text("run_id", "", "Pipeline run id (optional, UTC timestamp if empty)")

SCHEMA_NAME = dbutils.widgets.get("schema_name").strip()
SOURCE_BASE_PATH_WIDGET = dbutils.widgets.get("source_base_path").strip()
GENERATE_SAMPLE_DATA = dbutils.widgets.get("generate_sample_data") == "true"
CATALOG = dbutils.widgets.get("catalog").strip() or None
RUN_ID = dbutils.widgets.get("run_id").strip() or None

if "/FileStore/" in SOURCE_BASE_PATH_WIDGET:
    print(
        f"WARNING: Ignoring legacy FileStore path '{SOURCE_BASE_PATH_WIDGET}'. "
        "Will use {REPO_ROOT}/data after repo path is resolved."
    )
    SOURCE_BASE_PATH_WIDGET = ""

# SOURCE_BASE_PATH is finalized in the next section after REPO_ROOT is known.
SOURCE_BASE_PATH = SOURCE_BASE_PATH_WIDGET

print(f"schema={SCHEMA_NAME}")
print(f"source_base_path_widget={SOURCE_BASE_PATH_WIDGET or '(auto: repo data/)'}")
print(f"generate_sample_data={GENERATE_SAMPLE_DATA}")
print(f"catalog={CATALOG or '(default)'}")

## 2. Attach repository `src` to Python path

Resolves the repo root from this notebook's path under `/Workspace/Repos/...`.

In [ ]:
import os
import sys
from pathlib import Path

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

# e.g. /Repos/user@domain/databricks-medallion-pipeline/notebooks/run_full_pipeline
repo_rel = os.path.dirname(os.path.dirname(notebook_path))
REPO_ROOT = f"/Workspace{repo_rel}" if not repo_rel.startswith("/Workspace") else repo_rel
SRC_ROOT = os.path.join(REPO_ROOT, "src")
DATA_DIR = Path(REPO_ROOT) / "data"

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

assert os.path.isdir(SRC_ROOT), f"src not found at {SRC_ROOT} — open this notebook from the Git repo"

# Default ingest path: repo data/ (works on serverless when public DBFS FileStore is disabled)
if not SOURCE_BASE_PATH_WIDGET:
    SOURCE_BASE_PATH = DATA_DIR.resolve().as_uri()
elif SOURCE_BASE_PATH_WIDGET.startswith("/tmp") or SOURCE_BASE_PATH_WIDGET.startswith("file:/tmp"):
    print(
        f"WARNING: '{SOURCE_BASE_PATH_WIDGET}' is not readable on serverless; "
        f"using {DATA_DIR.resolve().as_uri()} instead."
    )
    SOURCE_BASE_PATH = DATA_DIR.resolve().as_uri()
else:
    SOURCE_BASE_PATH = SOURCE_BASE_PATH_WIDGET

print(f"REPO_ROOT={REPO_ROOT}")
print(f"SRC_ROOT={SRC_ROOT}")
print(f"source_base_path={SOURCE_BASE_PATH}")

## 3. (Optional) Generate sample CSVs into repo `data/`

Skip this section if CSVs are already at `source_base_path`.

In [ ]:
if GENERATE_SAMPLE_DATA:
    from data_generation.generate_sample_data import write_sample_datasets, DEFAULT_SEED

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Generating seed={DEFAULT_SEED} CSVs to {DATA_DIR} ...")
    write_sample_datasets(DATA_DIR, seed=DEFAULT_SEED)

    SOURCE_BASE_PATH = DATA_DIR.resolve().as_uri()
    print(f"Ingest will read from workspace path: {SOURCE_BASE_PATH}")
else:
    print(f"Skipping sample data generation; expecting CSVs at {SOURCE_BASE_PATH}")

## 4. Run end-to-end pipeline

Stages: config validation → Bronze ingest → Silver DQ → Gold build → reconciliation → final checks.

In [ ]:
import logging

from config.pipeline_config import load_config
from run_pipeline import configure_logging, run_pipeline

configure_logging("INFO")

config = load_config(
    source_base_path=SOURCE_BASE_PATH,
    catalog=CATALOG,
    schema_name=SCHEMA_NAME,
    run_id=RUN_ID,
)

print(f"Pipeline will ingest from: {config.source_base_path}")

summary = run_pipeline(
    config,
    spark=spark,
    repo_root=REPO_ROOT,
    generate_sample_data_flag=False,
    validate_sample_data=True,
    strict_sample_row_counts=False,
)

print("\n=== Pipeline summary ===")
print(f"run_id: {summary.run_id}")
print(f"batch_id: {summary.batch_id}")
print(f"elapsed_seconds: {summary.elapsed_seconds:.1f}")
print(f"steps: {summary.steps_completed}")
print(f"bronze_row_counts: {summary.bronze_row_counts}")
print(f"silver_row_counts: {summary.silver_row_counts}")
print(f"gold_row_counts: {summary.gold_row_counts}")

## 5. Quick validation (SQL)

Expected row counts for seed **42**: Bronze orders **100,000**; Gold products **500**; segmentation **4** segments.

In [ ]:
spark.sql(f"USE {SCHEMA_NAME}")

display(spark.sql("""
    SELECT 'bronze_customers' AS table_name, COUNT(*) AS row_count FROM bronze_customers
    UNION ALL SELECT 'bronze_products', COUNT(*) FROM bronze_products
    UNION ALL SELECT 'bronze_orders', COUNT(*) FROM bronze_orders
    UNION ALL SELECT 'silver_orders', COUNT(*) FROM silver_orders
    UNION ALL SELECT 'gold_sales_by_product', COUNT(*) FROM gold_sales_by_product
    UNION ALL SELECT 'gold_customer_segmentation', COUNT(*) FROM gold_customer_segmentation
"""))

display(spark.sql("""
    SELECT check_name, failed_records, pass_percentage
    FROM silver_dq_report
    WHERE failed_records > 0
    ORDER BY failed_records DESC
"""))

## 6. Next steps

1. Open `src/dashboard/dashboard_queries.sql` in the repo.
2. Follow **`src/dashboard/DASHBOARD_GUIDE.md`** to create the Databricks SQL Dashboard.
3. After `git push` from your laptop, **Pull** the repo in Databricks to sync changes.